### 1. 사용 내역 데이터 전처리

In [2]:
import os
import glob
import pandas as pd
import numpy as np

In [3]:
USAGE_DIR = "raw_data/usage"
REPAIR_DIR = "raw_data/repair"

usage_files = sorted(glob.glob(os.path.join(USAGE_DIR, "*.csv")))
repair_files = sorted(glob.glob(os.path.join(REPAIR_DIR, "*.csv")))

print("usage 파일 수:", len(usage_files))
print("repair 파일 수:", len(repair_files))

usage 파일 수: 59
repair 파일 수: 11


In [4]:
# 누락 컬럼 확인
keep_cols = ["자전거번호", "대여일시", "반납일시", "이용시간", "이용거리"]

for file in usage_files:
    df = pd.read_csv(file, encoding="cp949", low_memory=False, nrows=5)
    missing_cols = [col for col in keep_cols if col not in df.columns]
    
    if missing_cols:
        print("파일:", os.path.basename(file))
        print("누락 컬럼:", missing_cols)
        print("실제 컬럼:", df.columns.tolist())
        print("-" * 80)

파일: 2207.csv
누락 컬럼: ['이용시간', '이용거리']
실제 컬럼: ['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호', '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID', '반납대여소ID']
--------------------------------------------------------------------------------
파일: 2208.csv
누락 컬럼: ['이용시간', '이용거리']
실제 컬럼: ['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호', '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID', '반납대여소ID']
--------------------------------------------------------------------------------
파일: 2209.csv
누락 컬럼: ['이용시간', '이용거리']
실제 컬럼: ['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호', '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID', '반납대여소ID']
--------------------------------------------------------------------------------
파일: 2210.csv
누락 컬럼: ['이용시간', '이용거리']
실제 컬럼: ['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호', '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', 

In [5]:
# usage에서 남길 컬럼
keep_cols = ["자전거번호", "대여일시", "반납일시", "이용시간", "이용거리"]

summary_list = []

for i, file in enumerate(usage_files):
    print(f"[{i+1}/{len(usage_files)}] 처리 중:", os.path.basename(file))

    df = pd.read_csv(file, encoding="cp949", low_memory=False)

    # 컬럼명 통일
    df = df.rename(columns={
    "이용시간(분)": "이용시간",
    "이용거리(M)": "이용거리",
    })
    
    df_cut = df[keep_cols].copy()

    # 데이터 타입 정리
    df_cut["자전거번호"] = df_cut["자전거번호"].astype(str).str.strip()
    df_cut["대여일시"] = pd.to_datetime(df_cut["대여일시"], errors="coerce")
    df_cut["반납일시"] = pd.to_datetime(df_cut["반납일시"], errors="coerce")
    df_cut["이용시간"] = pd.to_numeric(df_cut["이용시간"], errors="coerce")
    df_cut["이용거리"] = pd.to_numeric(df_cut["이용거리"], errors="coerce")

    # 핵심 컬럼 결측치 제거
    df_cut = df_cut.dropna(subset=["자전거번호", "대여일시", "반납일시"])

    # 날짜, 시간 컬럼 생성
    df_cut["date"] = df_cut["대여일시"].dt.date
    df_cut["hour"] = df_cut["대여일시"].dt.hour

    # 일별 요약
    summary = (
        df_cut.groupby(["자전거번호", "date", "hour"], as_index=False)
        .agg(
            시간별_총이용시간=("이용시간", "sum"), # 사용량 확인
            시간별_총이용거리=("이용거리", "sum"), # 사용량 확인
            시간별_이용횟수=("자전거번호", "size"), # 사용 빈도 확인
            # 첫_대여시각=("대여일시", "min"),
            # 마지막_반납시각=("반납일시", "max")
        )
    )

    # 전처리 된 각 데이터 합치기
    summary_list.append(summary)

[1/59] 처리 중: 2102.csv
[2/59] 처리 중: 2103.csv
[3/59] 처리 중: 2104.csv
[4/59] 처리 중: 2105.csv
[5/59] 처리 중: 2106.csv
[6/59] 처리 중: 2107.csv
[7/59] 처리 중: 2108.csv
[8/59] 처리 중: 2109.csv
[9/59] 처리 중: 2110.csv
[10/59] 처리 중: 2111.csv
[11/59] 처리 중: 2112.csv
[12/59] 처리 중: 2201.csv
[13/59] 처리 중: 2202.csv
[14/59] 처리 중: 2203.csv
[15/59] 처리 중: 2204.csv
[16/59] 처리 중: 2205.csv
[17/59] 처리 중: 2206.csv
[18/59] 처리 중: 2207.csv
[19/59] 처리 중: 2208.csv
[20/59] 처리 중: 2209.csv
[21/59] 처리 중: 2210.csv
[22/59] 처리 중: 2211.csv
[23/59] 처리 중: 2212.csv
[24/59] 처리 중: 2301.csv
[25/59] 처리 중: 2302.csv
[26/59] 처리 중: 2303.csv
[27/59] 처리 중: 2304.csv
[28/59] 처리 중: 2305.csv
[29/59] 처리 중: 2306.csv
[30/59] 처리 중: 2307.csv
[31/59] 처리 중: 2308.csv
[32/59] 처리 중: 2309.csv
[33/59] 처리 중: 2310.csv
[34/59] 처리 중: 2311.csv
[35/59] 처리 중: 2312.csv
[36/59] 처리 중: 2401.csv
[37/59] 처리 중: 2402.csv
[38/59] 처리 중: 2403.csv
[39/59] 처리 중: 2404.csv
[40/59] 처리 중: 2405.csv
[41/59] 처리 중: 2406.csv
[42/59] 처리 중: 2407.csv
[43/59] 처리 중: 2408.csv
[44/59] 처리 중: 2409.c

In [6]:
# 전체 코드 합치기
usage_all = pd.concat(summary_list, ignore_index=True)

display(usage_all)
print("전체 shape:", usage_all.shape)

,자전거번호,date,hour,시간별_총이용시간,시간별_총이용거리,시간별_이용횟수
0,SPB-00385,2021-02-01,11,3.0,560.00,1
1,SPB-00385,2021-02-01,16,14.0,2320.00,1
2,SPB-00385,2021-02-01,17,7.0,2000.00,1
3,SPB-00385,2021-02-01,23,5.0,1230.00,1
4,SPB-00385,2021-02-02,6,7.0,1380.00,1
...,...,...,...,...,...,...
158381692,SPB-85099,2025-12-24,7,13.0,1679.56,1
158381693,SPB-85099,2025-12-31,13,5.0,600.35,1
158381694,SPB-85100,2025-12-17,11,25.0,3784.31,1
158381695,SPB-85100,2025-12-18,13,4.0,724.63,1


전체 shape: (158381697, 6)


In [7]:
import sys
print(sys.executable)

/usr/local/bin/python3


In [8]:
%pip install pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# usage_all의 date 컬럼 타입 수정
usage_all = usage_all.copy()

usage_all["date"] = pd.to_datetime(usage_all["date"], errors="coerce")
usage_all["hour"] = pd.to_numeric(usage_all["hour"], errors="coerce")
usage_all["시간별_총이용시간"] = pd.to_numeric(usage_all["시간별_총이용시간"], errors="coerce")
usage_all["시간별_총이용거리"] = pd.to_numeric(usage_all["시간별_총이용거리"], errors="coerce")
usage_all["시간별_이용횟수"] = pd.to_numeric(usage_all["시간별_이용횟수"], errors="coerce")

usage_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158381697 entries, 0 to 158381696
Data columns (total 6 columns):
 #   Column     Dtype         
---  ------     -----         
 0   자전거번호      object        
 1   date       datetime64[ns]
 2   hour       int32         
 3   시간별_총이용시간  float64       
 4   시간별_총이용거리  float64       
 5   시간별_이용횟수   int64         
dtypes: datetime64[ns](1), float64(2), int32(1), int64(1), object(1)
memory usage: 6.5+ GB


KeyboardInterrupt: 

In [ ]:
# 사용내역 전처리 데이터 저장
usage_all.to_parquet("processed_data/usage_all.parquet", index=False)

NameError: name 'usage_all' is not defined

In [8]:
# repair 데이터 샘플 확인
repair_sample = pd.read_csv(repair_files[0], encoding="cp949", low_memory=False) if repair_files else pd.DataFrame()
print(repair_sample.shape)
print(repair_sample.columns.tolist())
repair_sample.head()

(53843, 3)
['자전거번호', '등록일시', '고장구분']


,자전거번호,등록일시,고장구분
0,SPB-51735,2021-2-1,타이어
1,SPB-52819,2021-2-1,기타
2,SPB-53829,2021-2-1,기타
3,SPB-51114,2021-2-1,단말기
4,SPB-43267,2021-2-1,안장
